# 03 — Statistical Tests

Friedman test, Nemenyi post-hoc, pairwise Wilcoxon, Cohen's d effect sizes, and critical difference diagrams.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation.statistical_tests import (
    friedman_test, nemenyi_test, pairwise_wilcoxon,
    compute_average_ranks, plot_cd_diagram, pairwise_cohens_d,
)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

SAVE_DPI = 300
os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

# Load primary metrics from config
with open('../configs/experiment.yaml', 'r') as f:
    exp_cfg = yaml.safe_load(f)

PRIMARY_METRICS = {
    'binary': exp_cfg['primary_metric_binary'],        # roc_auc
    'multiclass': exp_cfg['primary_metric_multiclass'], # log_loss
    'regression': exp_cfg['primary_metric_regression'], # rmse
}
HIGHER_IS_BETTER = {
    'binary': True,
    'multiclass': False,  # log_loss: lower is better
    'regression': False,  # rmse: lower is better
}

print("Primary metrics:", PRIMARY_METRICS)

In [ ]:
# Load test results — graceful handling if files not yet generated
try:
    test_df = pd.read_csv('../results/aggregated/test_results.csv')
    print(f"Loaded test results: {test_df.shape}")
    print("Columns:", test_df.columns.tolist())
except FileNotFoundError as e:
    print(f"Results not yet available: {e}")
    print("Run: python scripts/run_all.py && python scripts/evaluate.py")
    test_df = pd.DataFrame()

## Friedman Test Results

The Friedman test is a non-parametric test for differences across multiple treatments (models) over multiple blocks (datasets). A low p-value means at least one model significantly differs from the rest.

In [ ]:
# Friedman test for each task type
tasks_config = [
    ('binary', PRIMARY_METRICS['binary'], HIGHER_IS_BETTER['binary']),
    ('multiclass', PRIMARY_METRICS['multiclass'], HIGHER_IS_BETTER['multiclass']),
    ('regression', PRIMARY_METRICS['regression'], HIGHER_IS_BETTER['regression']),
]

if not test_df.empty:
    friedman_rows = []

    for task_type, metric, higher_is_better in tasks_config:
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            print(f"Skipping {task_type}: no data for metric '{metric}'")
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)

        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            print(f"Skipping {task_type}: not enough data (need >=3 datasets and >=3 models), "
                  f"got {pivot.shape[0]} datasets x {pivot.shape[1]} models")
            continue

        friedman = friedman_test(pivot)
        ranks = compute_average_ranks(pivot, higher_is_better=higher_is_better)
        best_model = ranks.index[0]

        friedman_rows.append({
            'Task': task_type,
            'Metric': metric,
            'Higher=Better': higher_is_better,
            'N Datasets': pivot.shape[0],
            'N Models': pivot.shape[1],
            'Friedman chi2': round(friedman['statistic'], 3),
            'p-value': round(friedman['p_value'], 6),
            'Reject H0 (alpha=0.05)': friedman['reject_null'],
            'Best Model (avg rank)': best_model,
            'Best Avg Rank': round(ranks.iloc[0], 2),
        })

        print(f"\n{'='*60}")
        print(f"{task_type.upper()} — {metric} ({'higher' if higher_is_better else 'lower'} is better)")
        print(f"{'='*60}")
        print(f"Datasets: {pivot.shape[0]}, Models: {pivot.shape[1]}")
        print(f"Friedman: chi2={friedman['statistic']:.3f}, p={friedman['p_value']:.6f}, "
              f"reject_H0={friedman['reject_null']}")
        print(f"Average ranks:")
        for model, rank in ranks.items():
            print(f"  {model}: {rank:.3f}")

    if friedman_rows:
        friedman_table = pd.DataFrame(friedman_rows)
        print("\n=== Friedman Test Summary ===")
        display(friedman_table.style.set_caption(
            'Table: Friedman Test Results').hide(axis='index'))
else:
    print("Skipping: results not available.")

## Critical Difference Diagrams

CD diagrams show average ranks of models. A bar connecting models indicates they are NOT significantly different (Nemenyi post-hoc test at alpha=0.05).

In [ ]:
if not test_df.empty:
    for task_type, metric, higher_is_better in tasks_config:
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            continue

        save_path = f'../results/figures/cd_diagram_{task_type}.png'
        try:
            fig = plot_cd_diagram(
                pivot,
                title=f'Critical Difference — {task_type.title()} ({metric})',
                higher_is_better=higher_is_better,
                save_path=save_path,
            )
            # PDF is saved inside plot_cd_diagram via path.replace('.png', '.pdf')
            plt.show()
        except Exception as e:
            print(f"Could not plot CD diagram for {task_type}: {e}")
else:
    print("Skipping: results not available.")

## Pairwise Wilcoxon Tests

Pairwise Wilcoxon signed-rank tests with Holm-Bonferroni correction. Cells with p < 0.05 indicate statistically significant differences between model pairs.

In [ ]:
if not test_df.empty:
    for task_type, metric, higher_is_better in tasks_config:
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            continue

        try:
            p_values = pairwise_wilcoxon(pivot)

            plt.figure(figsize=(10, 8))
            mask = np.triu(np.ones_like(p_values, dtype=bool))
            sns.heatmap(p_values, annot=True, fmt='.4f', cmap='RdYlGn_r',
                        mask=mask, vmin=0, vmax=0.1, linewidths=0.5)
            plt.title(f'Pairwise Wilcoxon p-values (Holm-Bonferroni) — {task_type.title()} ({metric})')
            plt.tight_layout()
            fname = f'../results/figures/wilcoxon_{task_type}'
            plt.savefig(fname + '.png', dpi=SAVE_DPI, bbox_inches='tight')
            plt.savefig(fname + '.pdf', bbox_inches='tight')
            plt.show()
        except Exception as e:
            print(f"Could not compute Wilcoxon for {task_type}: {e}")
else:
    print("Skipping: results not available.")

## Nemenyi Post-Hoc Test

The Nemenyi test provides pairwise p-values after a significant Friedman test.

In [ ]:
if not test_df.empty:
    for task_type, metric, higher_is_better in tasks_config:
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            continue

        try:
            nemenyi = nemenyi_test(pivot, higher_is_better=higher_is_better)
            print(f"\n=== Nemenyi post-hoc: {task_type.upper()} ({metric}) ===")
            display(nemenyi.round(4).style.set_caption(
                f'Nemenyi p-values — {task_type.title()} ({metric})').background_gradient(
                cmap='RdYlGn', vmin=0, vmax=0.1))
        except Exception as e:
            print(f"Could not compute Nemenyi for {task_type}: {e}")
else:
    print("Skipping: results not available.")

## Cohen's d Effect Size Heatmap

Cohen's d measures the magnitude of the performance difference between model pairs (using paired samples across datasets). Larger absolute values indicate more practically significant differences.

In [ ]:
if not test_df.empty:
    for task_type, metric, higher_is_better in tasks_config:
        subset = test_df[test_df['task_type'] == task_type]
        if subset.empty or metric not in subset.columns:
            continue

        pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            continue

        try:
            d_matrix = pairwise_cohens_d(pivot)

            plt.figure(figsize=(10, 8))
            sns.heatmap(
                d_matrix,
                annot=True,
                fmt='.2f',
                cmap='RdBu_r',
                center=0,
                linewidths=0.5,
                vmin=-2,
                vmax=2,
            )
            plt.title(f"Pairwise Cohen's d Effect Size — {task_type.title()} ({metric})\n"
                      "Positive: row model > col model; Negative: row model < col model")
            plt.tight_layout()
            fname = f'../results/figures/cohens_d_{task_type}'
            plt.savefig(fname + '.png', dpi=SAVE_DPI, bbox_inches='tight')
            plt.savefig(fname + '.pdf', bbox_inches='tight')
            plt.show()

            # Interpretation guide
            print(f"Effect size interpretation (Cohen's d):")
            print("  |d| < 0.2  = negligible, 0.2-0.5 = small, 0.5-0.8 = medium, > 0.8 = large")
        except Exception as e:
            print(f"Could not compute Cohen's d for {task_type}: {e}")
else:
    print("Skipping: results not available.")

## Notes on Statistical Limitations

**Bootstrap CI limitations:** With only 10 datasets total (5 classification + 5 regression) in this benchmark, bootstrap confidence intervals have limited precision. The effective sample size for across-dataset tests is small, which reduces statistical power. Interpret p-values and effect sizes with caution — a result that fails to reach significance here may be practically important given the small benchmark size.

**Multiple comparisons:** With 8 models and 3 task groups, we apply Holm-Bonferroni correction to control familywise error rate. However, the Friedman/Nemenyi framework already accounts for multiple pairwise comparisons.

**Task heterogeneity:** Aggregating across binary, multiclass, and regression tasks conflates different problem types. Per-task analyses (above) are more reliable than overall rankings.